# 1. HDFS or S3? Start with the Workload

The useful question is not **which storage system is universally better?** The useful question is **which storage behavior does this workload need?**

- Choose **HDFS** when processing benefits from cluster-local distributed storage, low-latency filesystem operations, data locality, or frequent temporary I/O.
- Choose **Amazon S3** when data must remain durable beyond a cluster, be shared by multiple compute systems, and scale independently of compute.
- In many EMR designs, use both: **S3 for durable truth; HDFS for temporary working data.**

This presentation focuses on the decisions and engineering consequences behind that pattern.

# 2. Different Storage Models

HDFS is a **distributed filesystem**. Files are split into blocks, blocks are placed on DataNodes, and the NameNode tracks the namespace and block locations. Applications see filesystem-like paths and operations.

S3 is an **object store**. A bucket contains objects identified by keys. A key such as `curated/sales/date=2026-08-20/part-0001.parquet` looks like a path, but the slashes are part of the key; they do not create real directories.

This difference affects rename behavior, commit protocols, listing, small-file performance, recovery, and application design. A Hadoop connector translates filesystem calls into S3 API operations, but it cannot make S3 identical to HDFS.

# 3. Decision Matrix

| Requirement | Prefer HDFS | Prefer S3 |
|---|:---:|:---:|
| Data survives cluster replacement |  | ✓ |
| Data shared across many clusters/services |  | ✓ |
| Independent storage and compute scaling |  | ✓ |
| High-volume temporary shuffle/spill/cache | ✓ |  |
| Data locality is important | ✓ |  |
| Frequent filesystem renames | ✓ |  |
| Long-term data lake storage |  | ✓ |
| Short-lived intermediate results | ✓ | Possible, but often inefficient |
| Cluster-local application expects HDFS semantics | ✓ | Requires validation/adaptation |

Treat this as a starting point. File size, access pattern, framework, connector, security, recovery objective, and cost can change the decision.

# 4. When HDFS Is Still the Right Choice

A permanent or long-running cluster can legitimately rely on HDFS when:

- The application assumes a stable cluster and HDFS-native behavior
- Workloads repeatedly access the same hot data and benefit from locality
- Frequent metadata operations or renames are central to the workload
- Large intermediate datasets are reused across many jobs on that cluster
- Migration risk exceeds the current operational benefit

The trade-off is operational responsibility. Capacity, replication, NameNode health, disk failures, decommissioning, backups, and disaster recovery all require deliberate management. A long-running cluster is not the same as permanent storage: instances and disks can still fail or be replaced.

# 5. HDFS as Temporary Storage

On an ephemeral EMR cluster, HDFS is most valuable as a high-throughput workspace.

Good temporary uses:

- Cache repeatedly reused inputs during one processing run
- Hold intermediate MapReduce or Spark outputs
- Support shuffle, sort, spill, and checkpoint activity whose lifetime is limited to the job
- Stage data before producing fewer, better-sized output files

A safe lifecycle is:

`S3 source → HDFS working area → validate result → S3 destination → terminate cluster`

Never let the only copy of a valuable result remain in HDFS when the cluster may scale in or terminate.

# 6. A Little More About S3 Storage

S3 stores complete objects rather than mutable blocks. It provides strong read-after-write consistency for object `PUT`, `GET`, and `LIST` operations. Buckets can also use versioning, lifecycle rules, replication, access logging, and multiple storage classes.

Important design habits:

- Prefer columnar formats such as Parquet or ORC for analytics
- Partition by fields commonly used for pruning, without creating excessive partitions
- Avoid huge numbers of tiny objects; compact them into appropriately sized files
- Use bucket policies, IAM roles, encryption, and public-access controls
- Use lifecycle policies to transition or expire data intentionally

S3 durability does not prevent logical mistakes. Versioning and tested recovery procedures help with accidental overwrite or deletion.

# 7. How Hadoop Reaches S3

Hadoop-compatible engines use a connector that maps filesystem operations to S3 requests. Common path forms include `s3://bucket/key` and `s3a://bucket/key`.

On Amazon EMR:

- **EMRFS** has historically provided EMR integration with S3.
- Starting with **EMR 7.10.0**, **S3A** is the default S3 connector for `s3://`, `s3n://`, and `s3a://` schemes across EMR deployments.
- Trino and Presto have connector-specific behavior and must be reviewed separately.

Do not select a connector by path spelling alone. Confirm the EMR release, resolved filesystem implementation, credentials, encryption settings, and output committer before migration.

# 8. Copying HDFS Data to S3

For bulk transfer, Hadoop **DistCp** performs a parallel distributed copy:

```bash
hadoop distcp hdfs:///warehouse/events/ s3://my-data-lake/warehouse/events/
```

For a small item, a filesystem copy can be enough:

```bash
hadoop fs -cp hdfs:///reports/daily.parquet s3://my-data-lake/reports/
```

A copy is only the first migration step. Verify object counts, total bytes, checksums where the tool and format permit, record counts, schema, partition layout, encryption, ownership, and downstream readability before deleting the HDFS source.

# 9. Porting Applications from HDFS Paths to S3

Replace hard-coded HDFS locations with configuration-driven URIs:

```text
Before: hdfs:///warehouse/orders/
After:  s3://company-lake/warehouse/orders/
```

Then examine every assumption:

1. Does the job rename directories to commit output?
2. Does it append to files or modify bytes in place?
3. Does it create many tiny files?
4. Are HDFS permissions embedded in the workflow? Replace them with IAM and S3 controls.
5. Is a Hive metastore location or table definition still pointing to HDFS?
6. Are temporary, checkpoint, and final-output paths clearly separated?

Test correctness and failure recovery, not only the successful path.

# 10. Rename and Commit Are the Main Trap

In HDFS, renaming within the same filesystem is usually a fast metadata operation. S3 has no native directory rename. A connector may implement rename by copying objects to new keys and deleting the originals. For a large output tree, that can be slow, costly, and vulnerable to partial failure.

Safer practices:

- Use an S3-aware output committer supported by the chosen engine and connector
- Write each run to a unique destination, then publish completion through metadata or a small marker
- Avoid workflows that repeatedly rename large prefixes
- Make reruns idempotent and clean up incomplete multipart uploads with lifecycle rules

Never assume an HDFS commit algorithm remains correct or efficient merely because the same API call accepts an S3 URI.

# 11. Migration Plan: Move Safely in Stages

1. **Inventory** datasets, sizes, owners, permissions, consumers, formats, and update patterns.
2. **Classify** each dataset as durable source/output, shared reference data, or temporary working data.
3. **Design S3 layout** including bucket, prefix, partitioning, file size, encryption, and lifecycle.
4. **Copy a representative subset** with DistCp and validate it.
5. **Adapt applications** for S3 paths, credentials, committers, and object-store semantics.
6. **Dual-run and reconcile** counts, aggregates, schemas, and downstream results.
7. **Cut over readers**, then writers, with a rollback window.
8. **Retire HDFS copies** only after retention and recovery requirements are satisfied.

Move durable data first. Leave genuine scratch data on HDFS; migrating temporary bytes may create cost without value.

# 12. Common Hybrid Patterns

**Batch transformation**

- Read source tables from S3
- Use local disk/HDFS for shuffle and intermediate stages
- Write partitioned Parquet results to S3

**Long-running legacy platform**

- Keep latency-sensitive active data in HDFS
- Export snapshots or completed partitions to S3
- Test new consumers against S3, then reduce HDFS dependency gradually

**Shared reference dataset**

- Keep the authoritative copy in S3
- Copy or cache it in HDFS only when repeated access makes that worthwhile
- Rebuild the cache rather than treating it as the source of truth

# 13. Final Rules of Thumb

- Keep **durable, shared, authoritative data in S3**.
- Use **HDFS for temporary, locality-sensitive, high-I/O working data**.
- A stable long-running cluster may continue using HDFS when its workload depends on HDFS semantics, but it still needs backup and recovery design.
- Porting is more than changing `hdfs://` to `s3://`: validate permissions, commit behavior, file layout, metadata, correctness, and recovery.
- Use DistCp for parallel bulk copy, validate before cutover, and never delete the source prematurely.
- Design object sizes and partitions deliberately; tiny-file problems move with the data.

Further reading: [EMR storage and filesystems](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-plan-file-systems.html), [EMR architecture](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-overview-arch.html), and [EMRFS-to-S3A migration](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-s3a-migrate.html).